# FlashInspector AI – Test Model on Video + Feedback Loop

Upload a video and run your trained YOLOv8 models on it, then **correct mistakes and retrain**.

Runs **both** models on every frame and combines the annotations:
- **Detection** (`best_detect.pt`) — bounding boxes for fire safety equipment (S1,2,5,6)
- **Segmentation** (`best_segment.pt`) — masks for extinguisher condition/occlusion (S3,4)

### Feedback loop (human-in-the-loop retraining)
After inference, you can **review detections frame-by-frame**, correct mistakes
(remove false positives, reclassify, add missed objects), and **fine-tune the model**
on your corrections. Run inference again to see the improvement — repeat until satisfied.

```
Infer → Review → Correct → Retrain → Re-infer → ...
```

**Steps:**
1. Check GPU → Install deps → Clone repo (models + videos from GitHub)
2. Run inference on video
3. Review detections & correct mistakes
4. Fine-tune model on corrections
5. Re-run inference with improved model — repeat from step 3

## 1. Check GPU

Go to **Runtime → Change runtime type → T4 GPU** if no GPU is detected.

In [ ]:
!nvidia-smi 2>/dev/null || echo "\n\u26a0\ufe0f No GPU detected. Go to Runtime \u2192 Change runtime type \u2192 T4 GPU."

## 2. Install dependencies & clone repo

In [ ]:
!pip install -q ultralytics opencv-python easyocr

import os
from pathlib import Path
from google.colab import userdata

# --- Private repo: use a GitHub Personal Access Token ---
# Store your token in Colab Secrets (key icon in left sidebar) as GITHUB_TOKEN
token = userdata.get('GITHUB_TOKEN')

repo_dir = Path("/content/fire")
project_dir = repo_dir / "flashinspector-ai"

# Clone fresh if repo dir missing or incomplete (stale from previous session)
if not project_dir.exists():
    if repo_dir.exists():
        !rm -rf /content/fire
    !git clone --depth 1 https://{token}@github.com/patrisiyarum/fire.git /content/fire
else:
    print("Repo already cloned.")

os.chdir(str(project_dir))
print("\nProject ready at:", Path.cwd())

## 3. Load models from repo

Loads models directly from the cloned repository:
- **`best.pt`** — detection model (falls back to this if specific models aren't found)
- **`best_detect.pt`** — detection (S1: equipment, S2: marking signs, S5: inspection tags, S6: fire class symbols)
- **`best_segment.pt`** — segmentation (S3/S4: extinguisher condition/occlusion)

If neither `best_detect.pt` nor `best_segment.pt` exist, the notebook uses `best.pt` as the detection model.
You can also upload models manually if needed.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

# --- Service classification for model classes ---
# Keywords that map model class names → FireSafetyNet services
SERVICE_KEYWORDS = {
    "S5-inspection_tag": ["tag", "inspection", "expir", "date", "maintenance", "servic"],
    "S6-fire_class": ["class", "symbol", "type_a", "type_b", "type_c", "type_d", "type_e", "type_f",
                       "class_a", "class_b", "class_c", "class_d", "class_e", "class_f",
                       "powder", "foam", "co2", "water", "wet_chemical"],
    "S2-marking": ["marking", "sign", "signage", "suppression"],
    "S1-equipment": ["extinguisher", "blanket", "detector", "call_point", "alarm", "sounder",
                     "exit", "dome", "flash", "light", "orb", "hydrant"],
}

# Single-letter fire class symbols: A=ordinary, B=liquids, C=electrical, D=metals, F=cooking
FIRE_CLASS_LETTERS = {"a", "b", "c", "d", "e", "f", "k"}

def classify_service(class_name):
    """Determine which FireSafetyNet service a class belongs to."""
    name_lower = class_name.lower().strip().replace(" ", "_").replace("-", "_")
    # Exact match: single-letter fire class symbols (A, B, C, D, E, F, K)
    if name_lower in FIRE_CLASS_LETTERS:
        return "S6-fire_class"
    # Numeric fire ratings from S6 dataset (e.g. "1", "13", "21B" → rating numbers)
    if name_lower.isdigit():
        return "S6-fire_class"
    for service, keywords in SERVICE_KEYWORDS.items():
        for kw in keywords:
            if kw in name_lower:
                return service
    return "unknown"

# --- Load models from the repo ---
detect_model = None
segment_model = None

# Try loading specific models first, then fall back to best.pt
model_candidates = [
    ("best_detect.pt", "detect"),
    ("best_segment.pt", "segment"),
    ("best.pt", "detect"),  # fallback
]

for model_file, expected_task in model_candidates:
    p = Path(model_file)
    if not p.exists():
        continue
    m = YOLO(str(p))
    t = m.overrides.get("task", "detect")
    if t == "segment" and segment_model is None:
        segment_model = m
        print(f"Segmentation model loaded: {p}")
        print(f"  Classes ({len(m.names)}):")
        for idx, cls in m.names.items():
            svc = classify_service(cls)
            print(f"    [{idx}] {cls}  -> {svc}")
        print()
    elif t != "segment" and detect_model is None:
        detect_model = m
        print(f"Detection model loaded: {p}")
        print(f"  Classes ({len(m.names)}):")
        for idx, cls in m.names.items():
            svc = classify_service(cls)
            print(f"    [{idx}] {cls}  -> {svc}")
        print()

# If nothing loaded from repo, allow manual upload as fallback
if detect_model is None and segment_model is None:
    print("No model files found in repo. Falling back to manual upload...")
    from google.colab import files
    try:
        up = files.upload()
        for name in up.keys():
            p = Path(name)
            m = YOLO(str(p))
            t = m.overrides.get("task", "detect")
            if t == "segment" and segment_model is None:
                segment_model = m
                print(f"Segmentation model loaded: {p}")
            elif detect_model is None:
                detect_model = m
                print(f"Detection model loaded: {p}")
    except Exception:
        pass

if detect_model is None and segment_model is None:
    raise FileNotFoundError("No models found. Add best.pt, best_detect.pt, or best_segment.pt to the repo.")

# Build service map from the loaded model(s)
detect_service_map = {}
if detect_model:
    for idx, cls in detect_model.names.items():
        detect_service_map[cls] = classify_service(cls)
seg_service_map = {}
if segment_model:
    for idx, cls in segment_model.names.items():
        seg_service_map[cls] = classify_service(cls)

# Summary
models_loaded = []
if detect_model:
    models_loaded.append("Detection")
if segment_model:
    models_loaded.append("Segmentation")
print(f"Models ready: {', '.join(models_loaded)}")

# Check for service coverage
has_s5 = any("S5" in v for v in detect_service_map.values())
has_s6 = any("S6" in v for v in detect_service_map.values())
has_s3s4 = any("S3" in v or "S4" in v for v in seg_service_map.values()) if segment_model else False
if has_s5:
    print("  S5 (inspection tags): will OCR detected tag regions for expiry dates")
if has_s6:
    print("  S6 (fire class symbols): will identify fire class symbols")
if has_s3s4:
    print("  S3/S4 (condition check): will flag blocked/non-compliant extinguishers")
if not has_s5 and not has_s6:
    print("\n  NOTE: No S5/S6 classes detected in model. If your model should detect")
    print("  inspection tags or fire class symbols, check the class names above.")
    print("  The model may use different naming — update SERVICE_KEYWORDS if needed.")

## 4. Load video from repo

Loads a video from the `videos/` directory in the repository.
Add your test videos to `flashinspector-ai/videos/` and push to GitHub.

If no videos are found in the repo, you can upload one manually.

In [ ]:
from pathlib import Path

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
videos_dir = Path("videos")

# Find all video files in the videos/ directory
video_files = sorted(
    [f for f in videos_dir.glob("*") if f.suffix.lower() in VIDEO_EXTENSIONS]
) if videos_dir.exists() else []

if video_files:
    print(f"Found {len(video_files)} video(s) in videos/ directory:\n")
    for i, v in enumerate(video_files):
        size_mb = v.stat().st_size / 1024 / 1024
        print(f"  [{i}] {v.name}  ({size_mb:.1f} MB)")

    # Use the first video by default, or change the index below
    selected = 0  # <-- change this index to pick a different video
    VIDEO_PATH = video_files[selected]
    print(f"\nUsing: {VIDEO_PATH} ({VIDEO_PATH.stat().st_size / 1024 / 1024:.1f} MB)")
else:
    print("No videos found in videos/ directory. Falling back to manual upload...")
    from google.colab import files
    uploaded = files.upload()
    VIDEO_PATH = Path(list(uploaded.keys())[0])
    print(f"\nUploaded: {VIDEO_PATH} ({VIDEO_PATH.stat().st_size / 1024 / 1024:.1f} MB)")

## 5. Run inference on video (both models)

Runs both detection and segmentation models on each frame and combines the annotations.
- **CONFIDENCE**: minimum detection confidence (0.0–1.0). Lower values detect more (including small tags).
- **FRAME_SKIP**: process every Nth frame (higher = faster, lower = more detailed)

**Service-aware processing:**
- **S1** (equipment): fire extinguishers, blankets, smoke detectors, call points → bounding boxes
- **S2** (marking): FSE marking signs → bounding boxes
- **S3/S4** (condition): segmentation masks → flags blocked/non-compliant extinguishers
- **S5** (inspection tags): detects tag region → **OCR reads the date** → checks if expired
- **S6** (fire class symbols): detects symbols → identifies fire classes on extinguisher labels

In [ ]:
import cv2
import numpy as np
import re
import time
from datetime import datetime, date
from pathlib import Path

# ── Settings ──────────────────────────────────────────────────────────
CONFIDENCE = 0.15       # lower than default to catch small tags/symbols
FRAME_SKIP = 5          # process every Nth frame
OCR_CONF_MIN = 0.40     # only OCR extinguisher labels above this confidence
OCR_COOLDOWN = 10       # skip OCR for same-region detections within N frames

# ── Class name consolidation ──────────────────────────────────────────
CLASS_MAP = {
    "right exit": "emergency_exit", "left exit": "emergency_exit",
    "Right Exit": "emergency_exit", "Left Exit": "emergency_exit",
    "Straight Exit": "emergency_exit", "straight exit": "emergency_exit",
    "Left-Right Exit": "emergency_exit", "left-right exit": "emergency_exit",
    "emergency exit": "emergency_exit", "Emergency Exit": "emergency_exit",
    "fire-extinguisher": "fire_extinguisher",
    "fire extinguisher": "fire_extinguisher",
    "Fire_Extinguisher": "fire_extinguisher",
}

def consolidate_class(name):
    return CLASS_MAP.get(name, name)

# ── OCR setup for inspection tag reading (S5) ────────────────────────
import easyocr
ocr_reader = easyocr.Reader(["en"], gpu=True, verbose=False)
print("OCR reader loaded for inspection tag reading.\n")

def preprocess_for_ocr(crop):
    """Enhance a cropped region for better OCR accuracy."""
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    if max(h, w) < 200:
        scale = 200 / max(h, w)
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY, 15, 4)
    return binary

def is_useful_text(text):
    """Filter out garbled OCR noise — keep text that looks meaningful."""
    t = text.strip()
    if len(t) < 2:
        return False
    alnum = sum(c.isalnum() for c in t)
    if alnum == 0:
        return False
    if alnum / len(t) < 0.5:
        return False
    has_vowel_or_digit = bool(re.search(r'[aeiouAEIOU0-9]', t))
    if not has_vowel_or_digit and len(t) > 3:
        return False
    return True

def bbox_key(bbox, grid=80):
    """Quantize a bbox center to a grid cell for cooldown tracking."""
    cx = int((bbox[0] + bbox[2]) / 2) // grid
    cy = int((bbox[1] + bbox[3]) / 2) // grid
    return (cx, cy)

def read_tag_region(frame, bbox, padding=5):
    """Crop a detected tag/label region, preprocess, and OCR it."""
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in bbox]
    x1, y1 = max(0, x1 - padding), max(0, y1 - padding)
    x2, y2 = min(w, x2 + padding), min(h, y2 + padding)
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return []
    processed = preprocess_for_ocr(crop)
    texts = ocr_reader.readtext(processed, detail=0, paragraph=False)
    return [t for t in texts if is_useful_text(t)]

def extract_label_region(frame, bbox):
    """Extract the label area of an extinguisher (center band, 30-80% height).

    Fire extinguisher labels are typically in the middle portion of the body,
    not at the top (handle/nozzle) or bottom (base). This crops just the
    label band for much cleaner OCR.
    """
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in bbox]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    box_h = y2 - y1
    box_w = x2 - x1
    label_y1 = y1 + int(box_h * 0.30)
    label_y2 = y1 + int(box_h * 0.80)
    label_x1 = x1 + int(box_w * 0.10)
    label_x2 = x2 - int(box_w * 0.10)
    crop = frame[label_y1:label_y2, label_x1:label_x2]
    if crop.size == 0:
        return []
    processed = preprocess_for_ocr(crop)
    texts = ocr_reader.readtext(processed, detail=0, paragraph=False)
    return [t for t in texts if is_useful_text(t)]

def parse_dates(texts):
    """Try to extract dates from OCR text lines. Returns list of (date_obj, raw_text)."""
    found = []
    date_patterns = [
        (r'(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{4})', "dmy4"),
        (r'(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{2})', "dmy2"),
        (r'(\d{4})[/\-.](\d{1,2})[/\-.](\d{1,2})', "ymd"),
        (r'(\w+)\s+(\d{4})', "month_year"),
        (r'(\d{1,2})[/\-.](\d{4})', "my"),
    ]
    for text in texts:
        for pattern, fmt in date_patterns:
            m = re.search(pattern, text)
            if m:
                try:
                    if fmt == "dmy4":
                        d = datetime.strptime(f"{m.group(1)}/{m.group(2)}/{m.group(3)}", "%d/%m/%Y").date()
                    elif fmt == "dmy2":
                        d = datetime.strptime(f"{m.group(1)}/{m.group(2)}/{m.group(3)}", "%d/%m/%y").date()
                    elif fmt == "ymd":
                        d = datetime.strptime(f"{m.group(1)}/{m.group(2)}/{m.group(3)}", "%Y/%m/%d").date()
                    elif fmt == "month_year":
                        for mfmt in ["%B %Y", "%b %Y"]:
                            try:
                                d = datetime.strptime(f"{m.group(1)} {m.group(2)}", mfmt).date()
                                break
                            except ValueError:
                                continue
                        else:
                            continue
                    elif fmt == "my":
                        d = datetime.strptime(f"{m.group(1)}/{m.group(2)}", "%m/%Y").date()
                    found.append((d, text.strip()))
                except ValueError:
                    continue
    return found

def check_expiry(dates):
    """Check if any parsed dates are expired. Returns (is_expired, latest_date, raw_text)."""
    if not dates:
        return None, None, None
    today = date.today()
    latest = max(dates, key=lambda x: x[0])
    is_expired = latest[0] < today
    return is_expired, latest[0], latest[1]

# ── Service-aware color scheme ────────────────────────────────────────
SERVICE_COLORS = {
    "S1-equipment": (0, 255, 0),      # green
    "S2-marking": (255, 165, 0),       # orange
    "S5-inspection_tag": (0, 255, 255),# yellow
    "S6-fire_class": (255, 0, 255),    # magenta
    "unknown": (200, 200, 200),        # gray
}

# ── Open video ────────────────────────────────────────────────────────
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video: {VIDEO_PATH.name}")
print(f"Resolution: {width}x{height}, FPS: {fps:.1f}, Total frames: {total_frames}")
print(f"Duration: {total_frames / fps:.1f}s")
active = []
if detect_model: active.append("Detection")
if segment_model: active.append("Segmentation")
print(f"Running: {' + '.join(active)}  (conf={CONFIDENCE})")
print(f"OCR: label conf>={OCR_CONF_MIN}, cooldown={OCR_COOLDOWN} frames")
print(f"Processing every {FRAME_SKIP} frame(s)...\n")

# ── Output video ──────────────────────────────────────────────────────
out_dir = Path("inference_results")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / f"result_{VIDEO_PATH.stem}.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_writer = cv2.VideoWriter(str(out_path), fourcc, fps / FRAME_SKIP, (width, height))

# ── Process frames ────────────────────────────────────────────────────
detect_detections = []
segment_detections = []
tag_readings = []       # S5: OCR results from inspection tags
expiry_results = []     # S5: parsed expiry dates
symbol_detections = []  # S6: fire class symbols found
ocr_last_frame = {}     # cooldown tracker: bbox_key → last frame OCR'd
ocr_skipped = 0         # count of OCR calls skipped by cooldown

frame_idx = 0
processed = 0
start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % FRAME_SKIP == 0:
        annotated = frame.copy()
        timestamp = round(frame_idx / fps, 2)

        # ── Segmentation model (condition/occlusion masks) ───────────
        if segment_model:
            seg_results = segment_model(frame, conf=CONFIDENCE, verbose=False)[0]
            annotated = seg_results.plot(img=annotated, boxes=False)
            for box in seg_results.boxes:
                raw_cls = seg_results.names[int(box.cls)]
                cls_name = consolidate_class(raw_cls)
                svc = seg_service_map.get(raw_cls, classify_service(raw_cls))
                segment_detections.append({
                    "frame": frame_idx, "timestamp_sec": timestamp,
                    "class": cls_name, "confidence": round(float(box.conf), 3),
                    "model": "segment", "service": svc,
                })

        # ── Detection model (S1, S2, S5, S6) ─────────────────────────
        if detect_model:
            det_results = detect_model(frame, conf=CONFIDENCE, verbose=False)[0]

            for box in det_results.boxes:
                bbox_float = [float(x) for x in box.xyxy[0].tolist()]
                bbox = [int(x) for x in box.xyxy[0].tolist()]
                raw_cls = det_results.names[int(box.cls)]
                cls_name = consolidate_class(raw_cls)
                conf_val = float(box.conf)
                svc = detect_service_map.get(raw_cls, classify_service(raw_cls))

                # Color by service
                color = SERVICE_COLORS.get(svc, SERVICE_COLORS["unknown"])

                # Draw bounding box
                cv2.rectangle(annotated, (bbox[0], bbox[1]), (bbox[2], bbox[3]), color, 2)
                label = f"{cls_name} {conf_val:.2f}"
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
                cv2.rectangle(annotated, (bbox[0], bbox[1] - th - 6), (bbox[0] + tw, bbox[1]), color, -1)
                cv2.putText(annotated, label, (bbox[0], bbox[1] - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)

                detect_detections.append({
                    "frame": frame_idx, "timestamp_sec": timestamp,
                    "class": cls_name, "confidence": round(conf_val, 3),
                    "model": "detect", "bbox": bbox_float, "service": svc,
                })

                # ── S5: Inspection tag → OCR for expiry date ─────────
                if "S5" in svc:
                    bk = bbox_key(bbox_float)
                    if frame_idx - ocr_last_frame.get(("s5", bk), -999) >= OCR_COOLDOWN:
                        ocr_last_frame[("s5", bk)] = frame_idx
                        texts = read_tag_region(frame, bbox_float)
                        if texts:
                            tag_readings.append({
                                "frame": frame_idx, "timestamp_sec": timestamp,
                                "texts": texts, "bbox": bbox_float,
                            })
                            dates = parse_dates(texts)
                            is_expired, exp_date, raw = check_expiry(dates)
                            if is_expired is not None:
                                expiry_results.append({
                                    "frame": frame_idx, "timestamp_sec": timestamp,
                                    "expired": is_expired, "date": exp_date, "raw": raw,
                                })
                                status_color = (0, 0, 255) if is_expired else (0, 200, 0)
                                status_text = f"EXPIRED {exp_date}" if is_expired else f"Valid until {exp_date}"
                                cv2.putText(annotated, status_text, (bbox[0], bbox[3] + 20),
                                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, status_color, 2, cv2.LINE_AA)
                            else:
                                y_off = bbox[3] + 18
                                for txt in texts[:3]:
                                    cv2.putText(annotated, txt[:50], (bbox[0], y_off),
                                                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1, cv2.LINE_AA)
                                    y_off += 16
                    else:
                        ocr_skipped += 1

                # ── S6: Fire class symbol ─────────────────────────────
                if "S6" in svc:
                    symbol_detections.append({
                        "frame": frame_idx, "timestamp_sec": timestamp,
                        "class": cls_name, "confidence": round(conf_val, 3),
                    })
                    bk = bbox_key(bbox_float)
                    if frame_idx - ocr_last_frame.get(("s6", bk), -999) >= OCR_COOLDOWN:
                        ocr_last_frame[("s6", bk)] = frame_idx
                        sym_texts = read_tag_region(frame, bbox_float)
                        if sym_texts:
                            y_off = bbox[3] + 18
                            for txt in sym_texts[:2]:
                                cv2.putText(annotated, txt[:40], (bbox[0], y_off),
                                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 255), 1, cv2.LINE_AA)
                                y_off += 16
                    else:
                        ocr_skipped += 1

                # ── S1 equipment: OCR the label region of extinguishers
                if "extinguisher" in cls_name.lower() and "S1" in svc:
                    if conf_val >= OCR_CONF_MIN:
                        bk = bbox_key(bbox_float)
                        if frame_idx - ocr_last_frame.get(("ext", bk), -999) >= OCR_COOLDOWN:
                            ocr_last_frame[("ext", bk)] = frame_idx
                            texts = extract_label_region(frame, bbox_float)
                            if texts:
                                tag_readings.append({
                                    "frame": frame_idx, "timestamp_sec": timestamp,
                                    "texts": texts, "bbox": bbox_float, "source": "extinguisher_label",
                                })
                                dates = parse_dates(texts)
                                is_expired, exp_date, raw = check_expiry(dates)
                                if is_expired is not None:
                                    status_color = (0, 0, 255) if is_expired else (0, 200, 0)
                                    status_text = f"EXPIRED {exp_date}" if is_expired else f"Valid {exp_date}"
                                    cv2.putText(annotated, status_text, (bbox[0], bbox[3] + 20),
                                                cv2.FONT_HERSHEY_SIMPLEX, 0.55, status_color, 2, cv2.LINE_AA)
                                    expiry_results.append({
                                        "frame": frame_idx, "timestamp_sec": timestamp,
                                        "expired": is_expired, "date": exp_date, "raw": raw,
                                    })
                        else:
                            ocr_skipped += 1

        out_writer.write(annotated)
        processed += 1

        if processed % 50 == 0:
            elapsed = time.time() - start_time
            pct = frame_idx / total_frames * 100
            print(f"  {pct:.0f}% done — {processed} frames ({processed / elapsed:.1f} fps)")

    frame_idx += 1

cap.release()
out_writer.release()

elapsed = time.time() - start_time
all_detections = detect_detections + segment_detections
print(f"\nDone! Processed {processed} frames in {elapsed:.1f}s ({processed / max(elapsed, 0.01):.1f} fps)")
print(f"Annotated video saved: {out_path}")
if tag_readings:
    print(f"OCR: Read {len(tag_readings)} tag/label regions ({ocr_skipped} duplicate reads skipped by cooldown)")
if expiry_results:
    n_expired = sum(1 for e in expiry_results if e["expired"])
    print(f"Expiry check: {n_expired} expired, {len(expiry_results) - n_expired} valid")

## 6. Results — service-by-service breakdown + compliance report

Shows detections grouped by FireSafetyNet service, OCR readings from inspection tags, expiry status, and condition check results.

In [ ]:
from collections import Counter

def section(title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")

# ── Detection model by service ────────────────────────────────────────
if detect_detections:
    section("DETECTION MODEL — by service")
    by_svc = {}
    for d in detect_detections:
        svc = d.get("service", "unknown")
        by_svc.setdefault(svc, []).append(d)
    for svc in sorted(by_svc):
        dets = by_svc[svc]
        counts = Counter(d["class"] for d in dets)
        print(f"\n  [{svc}] — {len(dets)} detections")
        for cls, count in counts.most_common():
            confs = [d["confidence"] for d in dets if d["class"] == cls]
            print(f"    {cls}: {count} (avg conf: {sum(confs)/len(confs):.2f})")

# ── Segmentation model ───────────────────────────────────────────────
if segment_detections:
    section("SEGMENTATION MODEL — mask detections")
    counts = Counter(d["class"] for d in segment_detections)
    print(f"  Total: {len(segment_detections)} mask detections\n")
    for cls, count in counts.most_common():
        confs = [d["confidence"] for d in segment_detections if d["class"] == cls]
        print(f"    {cls}: {count} (avg conf: {sum(confs)/len(confs):.2f})")
    seg_frames = len(set(d["frame"] for d in segment_detections))
    print(f"\n  Masks drawn on {seg_frames} frames")

    seg_services = set(d.get("service", "") for d in segment_detections)
    has_condition_classes = any("S3" in s or "S4" in s for s in seg_services)
    if has_condition_classes:
        cond_dets = [d for d in segment_detections if "S3" in d.get("service","") or "S4" in d.get("service","")]
        section("S3/S4 — CONDITION CHECK FLAGS")
        cond_counts = Counter(d["class"] for d in cond_dets)
        for cls, count in cond_counts.most_common():
            print(f"    {cls}: {count} detections")
        cond_frames = len(set(d["frame"] for d in cond_dets))
        print(f"\n  Condition issues flagged in {cond_frames} frames")
    else:
        print(f"\n  NOTE: Segmentation model classes map to general equipment, not")
        print(f"  specific S3/S4 condition categories. The masks show extinguisher")
        print(f"  locations — for condition/occlusion flagging, train with S3/S4 classes")
        print(f"  (e.g. 'blocked', 'damaged', 'obstructed', 'non_compliant').")

# ── S5: Inspection tag OCR readings (deduplicated by frequency) ──────
section("S5 — INSPECTION TAG READINGS (OCR)")
if tag_readings:
    # Count how many times each unique text was read across all frames
    text_counts = Counter()
    text_first_seen = {}  # text → first timestamp
    for tr in tag_readings:
        src = tr.get("source", "inspection_tag")
        for txt in tr["texts"]:
            norm = txt.strip()
            if norm:
                text_counts[(norm, src)] += 1
                if (norm, src) not in text_first_seen:
                    text_first_seen[(norm, src)] = tr["timestamp_sec"]

    print(f"  Read text from {len(tag_readings)} tag/label regions")
    print(f"  Unique text strings: {len(text_counts)}\n")

    # Show most frequently read text first (more likely to be real)
    print("  Most consistent reads (sorted by frequency):\n")
    for (txt, src), count in text_counts.most_common(30):
        first_t = text_first_seen[(txt, src)]
        freq_bar = "#" * min(count, 20)
        print(f"    {count:>3}x  ({src}) \"{txt}\"  [first @ {first_t}s]  {freq_bar}")
else:
    print("  No inspection tag text detected.")
    print("  Possible reasons:")
    print("    - Model has no S5 (inspection tag) class — check class list above")
    print("    - Tags too small or blurry in video — try lower CONFIDENCE or higher resolution")
    print("    - Model wasn't trained on S5 data")

# ── S5: Expiry date check ────────────────────────────────────────────
section("S5 — EXPIRY DATE CHECK")
if expiry_results:
    from datetime import date
    today = date.today()
    for e in expiry_results:
        status = "EXPIRED" if e["expired"] else "VALID"
        icon = "[X]" if e["expired"] else "[OK]"
        print(f"  {icon} {status}: date={e['date']} (from \"{e['raw']}\") @ {e['timestamp_sec']}s")
    n_expired = sum(1 for e in expiry_results if e["expired"])
    n_valid = len(expiry_results) - n_expired
    print(f"\n  Summary: {n_expired} EXPIRED, {n_valid} VALID (checked against today: {today})")
else:
    print("  No dates found on tags/labels.")
    print("  OCR could not parse any date patterns from the detected regions.")

# ── S6: Fire class symbols ───────────────────────────────────────────
if symbol_detections:
    section("S6 — FIRE CLASS SYMBOLS")
    sym_counts = Counter(d["class"] for d in symbol_detections)
    for cls, count in sym_counts.most_common():
        print(f"    {cls}: detected {count} times")

# ── Combined totals ──────────────────────────────────────────────────
section("COMBINED TOTALS")
print(f"  Detection model: {len(detect_detections)} detections")
print(f"  Segmentation model: {len(segment_detections)} mask detections")
print(f"  Tag/label OCR reads: {len(tag_readings)}")
print(f"  Expiry dates found: {len(expiry_results)}")
print(f"  Fire class symbols: {len(symbol_detections)}")

if not detect_detections and not segment_detections:
    print(f"\n  No detections found. Try lowering CONFIDENCE (currently {CONFIDENCE}).")

## 7. Preview a sample frame

In [ ]:
import cv2
from IPython.display import display, Image as IPImage
from pathlib import Path
import tempfile

# Open the annotated video and grab a frame from the middle
cap = cv2.VideoCapture(str(out_path))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)  # jump to middle
ret, frame = cap.read()
cap.release()

if ret:
    tmp = Path(tempfile.mktemp(suffix=".jpg"))
    cv2.imwrite(str(tmp), frame)
    display(IPImage(str(tmp), width=800))
    print(f"Showing frame {total // 2} of {total}")
else:
    print("Could not read a frame from the output video.")

## 8. Download the annotated video

In [ ]:
from google.colab import files
from pathlib import Path

if Path(out_path).exists():
    print(f"Downloading: {out_path} ({Path(out_path).stat().st_size / 1024 / 1024:.1f} MB)")
    files.download(str(out_path))
else:
    print("Output video not found. Run the inference cell above first.")

---

# FEEDBACK LOOP — Correct Mistakes & Retrain

The cells below let you **review detections, correct mistakes, and fine-tune** the model.
Run through this cycle as many times as you like:

```
Review frames → Mark corrections → Export YOLO labels → Fine-tune → Re-infer
```

## 9. Extract review frames

Extracts key frames (those with the most detections) so you can review them efficiently.

In [ ]:
import cv2
import numpy as np
import json
from pathlib import Path
from collections import defaultdict

# ── Settings ──────────────────────────────────────────────────────────
MAX_REVIEW_FRAMES = 30   # max frames to extract for review

# ── Group detections by frame ─────────────────────────────────────────
frame_dets = defaultdict(list)
for d in detect_detections:
    frame_dets[d["frame"]].append(d)

# Sort frames by number of detections (most interesting first)
ranked_frames = sorted(frame_dets.keys(), key=lambda f: len(frame_dets[f]), reverse=True)
review_frame_ids = ranked_frames[:MAX_REVIEW_FRAMES]
review_frame_ids.sort()  # chronological order for review

print(f"Total frames with detections: {len(frame_dets)}")
print(f"Selected {len(review_frame_ids)} frames for review (most detections first)\n")

# ── Extract raw frames from video ────────────────────────────────────
review_dir = Path("feedback_review")
review_dir.mkdir(exist_ok=True)
(review_dir / "images").mkdir(exist_ok=True)

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

frame_set = set(review_frame_ids)
review_frames = {}  # frame_idx → image path
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx in frame_set:
        img_path = review_dir / "images" / f"frame_{frame_idx:06d}.jpg"
        cv2.imwrite(str(img_path), frame)
        review_frames[frame_idx] = img_path
    frame_idx += 1

cap.release()

# ── Build review data structure ──────────────────────────────────────
# Each review entry: {frame, image_path, detections: [{id, class, conf, bbox, action}]}
review_data = []
for fidx in review_frame_ids:
    if fidx not in review_frames:
        continue
    entry = {
        "frame": fidx,
        "image_path": str(review_frames[fidx]),
        "detections": []
    }
    for i, d in enumerate(frame_dets[fidx]):
        entry["detections"].append({
            "id": i,
            "class": d["class"],
            "confidence": d["confidence"],
            "bbox": d["bbox"],
            "service": d["service"],
            "action": "keep",       # default: keep
            "new_class": None,      # set if reclassified
        })
    review_data.append(entry)

# ── Save review data ─────────────────────────────────────────────────
review_json = review_dir / "review_data.json"
with open(review_json, "w") as f:
    json.dump(review_data, f, indent=2, default=str)

print(f"Extracted {len(review_frames)} frames to {review_dir}/images/")
print(f"Review data saved to {review_json}")
print(f"\nDetection counts per frame:")
for entry in review_data[:10]:
    n = len(entry["detections"])
    print(f"  Frame {entry['frame']}: {n} detection(s)")
if len(review_data) > 10:
    print(f"  ... and {len(review_data) - 10} more frames")

## 10. Interactive review — correct mistakes

Use the widget below to review each frame. For each detection you can:
- **Keep** — detection is correct (becomes positive training data)
- **Remove** — false positive / wrong (removed from training data)
- **Reclassify** — correct class is wrong, pick the right one from the dropdown

You can also **add missed detections** that the model failed to find.

In [ ]:
import cv2
import json
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display, clear_output
from pathlib import Path

# ── Build class list from model ───────────────────────────────────────
all_classes = []
if detect_model:
    all_classes = list(detect_model.names.values())
elif review_data:
    all_classes = list(set(d["class"] for entry in review_data for d in entry["detections"]))
all_classes = sorted(set(all_classes))

# ── Color map for boxes ──────────────────────────────────────────────
ACTION_COLORS = {"keep": "lime", "remove": "red", "reclassify": "cyan"}

# ── State ─────────────────────────────────────────────────────────────
added_boxes = {}  # frame_idx → list of {class, bbox_norm}

# ── Widget layout ─────────────────────────────────────────────────────
frame_slider = widgets.IntSlider(
    value=0, min=0, max=max(len(review_data) - 1, 0),
    description="Frame:", continuous_update=False,
    layout=widgets.Layout(width="600px"),
)
output_area = widgets.Output(layout=widgets.Layout(width="100%", min_height="500px"))
controls_area = widgets.Output(layout=widgets.Layout(width="100%"))
status_label = widgets.HTML(value="<b>Use the slider to navigate frames. Click buttons to correct detections.</b>")

# ── Add missed detection controls ────────────────────────────────────
add_class_dd = widgets.Dropdown(options=all_classes, description="Class:")
add_coords = widgets.Text(
    description="BBox (%):",
    placeholder="x1,y1,x2,y2 as % (e.g. 10,20,30,50)",
    layout=widgets.Layout(width="350px"),
)
add_btn = widgets.Button(description="Add missed detection", button_style="success")

def render_frame(idx):
    """Draw the frame with numbered bounding boxes colored by action."""
    if idx >= len(review_data):
        return
    entry = review_data[idx]
    img = cv2.imread(entry["image_path"])
    if img is None:
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    with output_area:
        clear_output(wait=True)
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
        ax.imshow(img_rgb)
        ax.set_title(f"Frame {entry['frame']}  ({idx + 1}/{len(review_data)})")
        ax.axis("off")

        for det in entry["detections"]:
            bbox = det["bbox"]
            x1, y1, x2, y2 = bbox
            color = ACTION_COLORS.get(det["action"], "white")
            cls_label = det.get("new_class") or det["class"]
            label = f"#{det['id']} {cls_label} ({det['confidence']:.2f}) [{det['action']}]"

            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                     linewidth=2, edgecolor=color, facecolor="none")
            ax.add_patch(rect)
            ax.text(x1, y1 - 5, label, fontsize=7, color=color,
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.7))

        # Draw added boxes
        fidx = entry["frame"]
        for ab in added_boxes.get(fidx, []):
            bx = [ab["bbox_norm"][i] * (w if i % 2 == 0 else h) for i in range(4)]
            rect = patches.Rectangle((bx[0], bx[1]), bx[2] - bx[0], bx[3] - bx[1],
                                     linewidth=2, edgecolor="yellow", facecolor="none", linestyle="--")
            ax.add_patch(rect)
            ax.text(bx[0], bx[1] - 5, f"[ADDED] {ab['class']}", fontsize=7, color="yellow",
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.7))

        plt.tight_layout()
        plt.show()

def render_controls(idx):
    """Create per-detection control buttons."""
    if idx >= len(review_data):
        return
    entry = review_data[idx]

    with controls_area:
        clear_output(wait=True)
        for det in entry["detections"]:
            cls_label = det.get("new_class") or det["class"]
            header = widgets.HTML(
                f"<b>#{det['id']}</b> {cls_label} (conf={det['confidence']:.2f}, {det['service']}) "
                f"— current: <span style='color:{ACTION_COLORS[det['action']]}'><b>{det['action']}</b></span>"
            )
            keep_btn = widgets.Button(description="Keep", button_style="success",
                                      layout=widgets.Layout(width="80px"))
            remove_btn = widgets.Button(description="Remove", button_style="danger",
                                        layout=widgets.Layout(width="80px"))
            reclass_dd = widgets.Dropdown(options=[""] + all_classes, value="",
                                          description="Reclassify:",
                                          layout=widgets.Layout(width="250px"))

            det_id = det["id"]
            frame_i = idx

            def make_keep(di=det_id, fi=frame_i):
                def on_keep(b):
                    review_data[fi]["detections"][di]["action"] = "keep"
                    review_data[fi]["detections"][di]["new_class"] = None
                    render_frame(fi)
                    render_controls(fi)
                return on_keep

            def make_remove(di=det_id, fi=frame_i):
                def on_remove(b):
                    review_data[fi]["detections"][di]["action"] = "remove"
                    render_frame(fi)
                    render_controls(fi)
                return on_remove

            def make_reclass(di=det_id, fi=frame_i):
                def on_reclass(change):
                    if change["new"]:
                        review_data[fi]["detections"][di]["action"] = "reclassify"
                        review_data[fi]["detections"][di]["new_class"] = change["new"]
                        render_frame(fi)
                        render_controls(fi)
                return on_reclass

            keep_btn.on_click(make_keep())
            remove_btn.on_click(make_remove())
            reclass_dd.observe(make_reclass(), names="value")

            display(widgets.HBox([header, keep_btn, remove_btn, reclass_dd]))

def on_add_missed(b):
    """Add a missed detection to the current frame."""
    idx = frame_slider.value
    entry = review_data[idx]
    fidx = entry["frame"]
    coords_str = add_coords.value.strip()
    cls = add_class_dd.value
    try:
        parts = [float(x.strip()) / 100.0 for x in coords_str.split(",")]
        assert len(parts) == 4, "Need 4 values"
        assert all(0 <= p <= 1 for p in parts), "Values must be 0-100"
    except Exception as e:
        status_label.value = f"<span style='color:red'>Invalid coords: {e}. Use x1,y1,x2,y2 as percentages (0-100).</span>"
        return
    if fidx not in added_boxes:
        added_boxes[fidx] = []
    added_boxes[fidx].append({"class": cls, "bbox_norm": parts})
    status_label.value = f"<span style='color:lime'>Added {cls} box to frame {fidx}.</span>"
    render_frame(idx)

add_btn.on_click(on_add_missed)

def on_frame_change(change):
    render_frame(change["new"])
    render_controls(change["new"])
    n_kept = sum(1 for e in review_data for d in e["detections"] if d["action"] == "keep")
    n_removed = sum(1 for e in review_data for d in e["detections"] if d["action"] == "remove")
    n_reclass = sum(1 for e in review_data for d in e["detections"] if d["action"] == "reclassify")
    n_added = sum(len(v) for v in added_boxes.values())
    status_label.value = (
        f"<b>Progress:</b> {n_kept} kept, {n_removed} removed, "
        f"{n_reclass} reclassified, {n_added} added"
    )

frame_slider.observe(on_frame_change, names="value")

# ── Display ───────────────────────────────────────────────────────────
add_section = widgets.HBox([add_class_dd, add_coords, add_btn])
display(widgets.VBox([
    status_label,
    frame_slider,
    output_area,
    controls_area,
    widgets.HTML("<hr><b>Add missed detection:</b> Enter bbox as percentages of image (x1,y1,x2,y2):"),
    add_section,
]))

# Initial render
render_frame(0)
render_controls(0)

## 11. Export corrections as YOLO training data

Converts your reviewed corrections into YOLO-format labels for fine-tuning:
- **Kept** detections become positive training examples
- **Reclassified** detections use the corrected class
- **Removed** detections are excluded (teaches the model not to detect those)
- **Added** detections become new positive examples

Creates a `feedback_dataset/` directory with `images/`, `labels/`, and `data.yaml`.

In [ ]:
import cv2
import json
import shutil
import yaml
from pathlib import Path

# ── Dataset directory structure ───────────────────────────────────────
ds_dir = Path("feedback_dataset")
if ds_dir.exists():
    shutil.rmtree(ds_dir)

train_img = ds_dir / "images" / "train"
train_lbl = ds_dir / "labels" / "train"
val_img = ds_dir / "images" / "val"
val_lbl = ds_dir / "labels" / "val"
for d in [train_img, train_lbl, val_img, val_lbl]:
    d.mkdir(parents=True)

# ── Build unified class list ─────────────────────────────────────────
# Use the detection model's class names as the base, add any new classes from corrections
class_names = list(detect_model.names.values()) if detect_model else []
# Add classes from added boxes that might not be in the model
for boxes in added_boxes.values():
    for ab in boxes:
        if ab["class"] not in class_names:
            class_names.append(ab["class"])
# Add reclassified classes
for entry in review_data:
    for det in entry["detections"]:
        if det["new_class"] and det["new_class"] not in class_names:
            class_names.append(det["new_class"])

class_to_id = {name: i for i, name in enumerate(class_names)}
print(f"Class mapping ({len(class_names)} classes):")
for name, idx in class_to_id.items():
    print(f"  [{idx}] {name}")

# ── Convert detections to YOLO format ─────────────────────────────────
n_images = 0
n_labels = 0
n_removed = 0
n_added = 0

val_split = 0.2  # 20% for validation

for i, entry in enumerate(review_data):
    img_path = Path(entry["image_path"])
    if not img_path.exists():
        continue

    img = cv2.imread(str(img_path))
    if img is None:
        continue
    h, w = img.shape[:2]

    # Determine train/val split
    is_val = (i % int(1 / val_split)) == 0
    dst_img_dir = val_img if is_val else train_img
    dst_lbl_dir = val_lbl if is_val else train_lbl

    img_name = img_path.name
    lbl_name = img_path.stem + ".txt"

    lines = []

    # Process model detections (kept + reclassified)
    for det in entry["detections"]:
        if det["action"] == "remove":
            n_removed += 1
            continue

        cls_name = det.get("new_class") or det["class"]
        if cls_name not in class_to_id:
            continue

        cls_id = class_to_id[cls_name]
        x1, y1, x2, y2 = det["bbox"]

        # Convert to YOLO format: class cx cy w h (normalized 0-1)
        cx = ((x1 + x2) / 2) / w
        cy = ((y1 + y2) / 2) / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h

        # Clamp to [0, 1]
        cx = max(0, min(1, cx))
        cy = max(0, min(1, cy))
        bw = max(0, min(1, bw))
        bh = max(0, min(1, bh))

        lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        n_labels += 1

    # Process manually added detections
    fidx = entry["frame"]
    for ab in added_boxes.get(fidx, []):
        cls_name = ab["class"]
        if cls_name not in class_to_id:
            continue
        cls_id = class_to_id[cls_name]
        nx1, ny1, nx2, ny2 = ab["bbox_norm"]
        cx = (nx1 + nx2) / 2
        cy = (ny1 + ny2) / 2
        bw = nx2 - nx1
        bh = ny2 - ny1
        lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        n_labels += 1
        n_added += 1

    # Copy image and write label file
    shutil.copy2(str(img_path), str(dst_img_dir / img_name))
    with open(dst_lbl_dir / lbl_name, "w") as f:
        f.write("\n".join(lines))
    n_images += 1

# ── Generate data.yaml ───────────────────────────────────────────────
data_yaml = {
    "path": str(ds_dir.resolve()),
    "train": "images/train",
    "val": "images/val",
    "nc": len(class_names),
    "names": class_names,
}
yaml_path = ds_dir / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

# ── Summary ───────────────────────────────────────────────────────────
n_train = len(list(train_img.glob("*.jpg")))
n_val = len(list(val_img.glob("*.jpg")))

print(f"\nFeedback dataset created at: {ds_dir}/")
print(f"  Images: {n_images} ({n_train} train, {n_val} val)")
print(f"  Labels: {n_labels} bounding boxes")
print(f"  Removed (false positives): {n_removed}")
print(f"  Added (missed detections): {n_added}")
print(f"  data.yaml: {yaml_path}")

# Save final review state
review_json = Path("feedback_review") / "review_data_final.json"
final_data = {
    "review_data": review_data,
    "added_boxes": {str(k): v for k, v in added_boxes.items()},
    "class_mapping": class_to_id,
}
with open(review_json, "w") as f:
    json.dump(final_data, f, indent=2, default=str)
print(f"  Review state saved: {review_json}")

## 12. Fine-tune model on corrections

Fine-tunes the detection model using your corrected data. This uses **transfer learning** —
starting from the current model weights and training for a few epochs on the feedback data.

Settings:
- **FINETUNE_EPOCHS**: more epochs = more learning from corrections, but risk overfitting
- **FREEZE_LAYERS**: freeze early layers to preserve general knowledge, only train detection heads
- **LR**: low learning rate to make small adjustments, not overwrite everything

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil

# ── Fine-tuning settings ─────────────────────────────────────────────
FINETUNE_EPOCHS = 15     # number of epochs (adjust: more = learns more, risk overfitting)
FREEZE_LAYERS = 10       # freeze first N backbone layers (preserves general features)
LEARNING_RATE = 0.001    # low LR for gentle fine-tuning
BATCH_SIZE = 8           # adjust based on GPU memory
IMG_SIZE = 640           # match your original training resolution

# ── Track feedback round ─────────────────────────────────────────────
feedback_round_file = Path("feedback_review") / ".round"
if feedback_round_file.exists():
    feedback_round = int(feedback_round_file.read_text().strip()) + 1
else:
    feedback_round = 1
feedback_round_file.write_text(str(feedback_round))

print(f"=== FEEDBACK ROUND {feedback_round} ===\n")

# ── Select base model for fine-tuning ────────────────────────────────
# Use the latest fine-tuned model if available, otherwise the original
finetuned_dir = Path("feedback_models")
finetuned_dir.mkdir(exist_ok=True)

prev_model_path = finetuned_dir / f"best_round_{feedback_round - 1}.pt"
if prev_model_path.exists():
    base_model_path = str(prev_model_path)
    print(f"Starting from previous fine-tuned model: {prev_model_path}")
elif detect_model:
    # Find the original model file path
    base_model_path = detect_model.ckpt_path if hasattr(detect_model, 'ckpt_path') else "best.pt"
    print(f"Starting from original model: {base_model_path}")
else:
    raise RuntimeError("No detection model available for fine-tuning.")

# ── Fine-tune ─────────────────────────────────────────────────────────
yaml_path = Path("feedback_dataset") / "data.yaml"
if not yaml_path.exists():
    raise FileNotFoundError("No feedback dataset found. Run the export cell (Section 11) first.")

print(f"\nFine-tuning with:")
print(f"  Dataset: {yaml_path}")
print(f"  Epochs: {FINETUNE_EPOCHS}")
print(f"  Frozen layers: {FREEZE_LAYERS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image size: {IMG_SIZE}")
print()

ft_model = YOLO(base_model_path)
results = ft_model.train(
    data=str(yaml_path),
    epochs=FINETUNE_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    lr0=LEARNING_RATE,
    lrf=0.01,             # final LR = lr0 * lrf
    freeze=FREEZE_LAYERS,
    patience=5,            # early stopping patience
    project="feedback_runs",
    name=f"round_{feedback_round}",
    exist_ok=True,
    verbose=True,
)

# ── Save fine-tuned model ────────────────────────────────────────────
best_pt = Path(f"feedback_runs/round_{feedback_round}/weights/best.pt")
if best_pt.exists():
    save_path = finetuned_dir / f"best_round_{feedback_round}.pt"
    shutil.copy2(str(best_pt), str(save_path))
    print(f"\nFine-tuned model saved: {save_path}")

    # Update detect_model to use the new weights
    detect_model = YOLO(str(save_path))
    print(f"Detection model updated to round {feedback_round} weights.")

    # Rebuild service map
    detect_service_map = {}
    for idx, cls in detect_model.names.items():
        detect_service_map[cls] = classify_service(cls)
else:
    print("\nWarning: Fine-tuning did not produce a best.pt. Check training output above.")

## 13. Re-run inference with improved model & compare

Runs inference again using the fine-tuned model and compares results side-by-side with
the original. Shows improvement metrics and a visual comparison.

**To continue the feedback loop:** go back to Section 9, re-run the extract/review/export/fine-tune
cells again — each round builds on the last.

In [ ]:
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

# ── Re-run inference on the same video with the updated model ─────────
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Store previous results for comparison
prev_detect_count = len(detect_detections)
prev_class_counts = Counter(d["class"] for d in detect_detections)

print(f"Re-running inference with fine-tuned model (round {feedback_round})...")
print(f"Video: {VIDEO_PATH.name}, {total_frames} frames\n")

# Output video
out_path_new = out_dir / f"result_{VIDEO_PATH.stem}_round{feedback_round}.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_writer = cv2.VideoWriter(str(out_path_new), fourcc, fps / FRAME_SKIP, (width, height))

new_detections = []
frame_idx = 0
processed = 0
start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % FRAME_SKIP == 0:
        annotated = frame.copy()
        timestamp = round(frame_idx / fps, 2)

        if detect_model:
            det_results = detect_model(frame, conf=CONFIDENCE, verbose=False)[0]
            for box in det_results.boxes:
                bbox = [int(x) for x in box.xyxy[0].tolist()]
                bbox_float = [float(x) for x in box.xyxy[0].tolist()]
                raw_cls = det_results.names[int(box.cls)]
                cls_name = consolidate_class(raw_cls)
                conf_val = float(box.conf)
                svc = detect_service_map.get(raw_cls, classify_service(raw_cls))
                color = SERVICE_COLORS.get(svc, SERVICE_COLORS["unknown"])

                cv2.rectangle(annotated, (bbox[0], bbox[1]), (bbox[2], bbox[3]), color, 2)
                label = f"{cls_name} {conf_val:.2f}"
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
                cv2.rectangle(annotated, (bbox[0], bbox[1] - th - 6), (bbox[0] + tw, bbox[1]), color, -1)
                cv2.putText(annotated, label, (bbox[0], bbox[1] - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)

                new_detections.append({
                    "frame": frame_idx, "timestamp_sec": timestamp,
                    "class": cls_name, "confidence": round(conf_val, 3),
                    "bbox": bbox_float, "service": svc,
                })

        out_writer.write(annotated)
        processed += 1

        if processed % 50 == 0:
            elapsed = time.time() - start_time
            pct = frame_idx / total_frames * 100
            print(f"  {pct:.0f}% done — {processed} frames")

    frame_idx += 1

cap.release()
out_writer.release()

elapsed = time.time() - start_time
print(f"\nDone! Processed {processed} frames in {elapsed:.1f}s")
print(f"New annotated video: {out_path_new}")

# ── Comparison ────────────────────────────────────────────────────────
new_class_counts = Counter(d["class"] for d in new_detections)

print(f"\n{'='*60}")
print(f"  COMPARISON: Before vs After Fine-tuning (Round {feedback_round})")
print(f"{'='*60}")
print(f"\n  Total detections: {prev_detect_count} -> {len(new_detections)}")
print(f"  Change: {len(new_detections) - prev_detect_count:+d}")

print(f"\n  Per-class breakdown:")
all_cls = sorted(set(list(prev_class_counts.keys()) + list(new_class_counts.keys())))
for cls in all_cls:
    before = prev_class_counts.get(cls, 0)
    after = new_class_counts.get(cls, 0)
    diff = after - before
    arrow = "+" if diff > 0 else ""
    print(f"    {cls}: {before} -> {after} ({arrow}{diff})")

# ── Confidence comparison ─────────────────────────────────────────────
if detect_detections and new_detections:
    prev_avg_conf = sum(d["confidence"] for d in detect_detections) / len(detect_detections)
    new_avg_conf = sum(d["confidence"] for d in new_detections) / len(new_detections)
    print(f"\n  Avg confidence: {prev_avg_conf:.3f} -> {new_avg_conf:.3f}")

# ── Visual comparison: sample frame side-by-side ──────────────────────
print(f"\n  Side-by-side comparison (middle frame):")

cap_old = cv2.VideoCapture(str(out_path))
cap_new = cv2.VideoCapture(str(out_path_new))
total_old = int(cap_old.get(cv2.CAP_PROP_FRAME_COUNT))
total_new = int(cap_new.get(cv2.CAP_PROP_FRAME_COUNT))
mid = min(total_old, total_new) // 2

cap_old.set(cv2.CAP_PROP_POS_FRAMES, mid)
cap_new.set(cv2.CAP_PROP_POS_FRAMES, mid)
ret_old, frame_old = cap_old.read()
ret_new, frame_new = cap_new.read()
cap_old.release()
cap_new.release()

if ret_old and ret_new:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    ax1.imshow(cv2.cvtColor(frame_old, cv2.COLOR_BGR2RGB))
    ax1.set_title("BEFORE (original model)")
    ax1.axis("off")
    ax2.imshow(cv2.cvtColor(frame_new, cv2.COLOR_BGR2RGB))
    ax2.set_title(f"AFTER (round {feedback_round})")
    ax2.axis("off")
    plt.tight_layout()
    plt.show()

# Update detect_detections for next feedback round
detect_detections = new_detections

print(f"\n  To continue improving: go back to Section 9 and repeat the feedback loop.")
print(f"  The model will keep learning from your corrections each round.")

## 14. Download improved model

Download the fine-tuned model to use in production or push back to your repo.

In [ ]:
from google.colab import files
from pathlib import Path

finetuned_dir = Path("feedback_models")
models = sorted(finetuned_dir.glob("best_round_*.pt"))

if models:
    latest = models[-1]
    print(f"Available fine-tuned models:")
    for m in models:
        size_mb = m.stat().st_size / 1024 / 1024
        print(f"  {m.name}  ({size_mb:.1f} MB)")

    print(f"\nDownloading latest: {latest.name}")
    files.download(str(latest))

    # Also download the new annotated video if it exists
    new_video = out_dir / f"result_{VIDEO_PATH.stem}_round{feedback_round}.mp4"
    if new_video.exists():
        print(f"Downloading annotated video: {new_video.name}")
        files.download(str(new_video))
else:
    print("No fine-tuned models found. Run the fine-tuning cell (Section 12) first.")